# 02 – Feature Analysis

Feature importance, variance filtering, delta features, and dimensionality reduction
for the EmoPairCompete dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

from src.data_loader import load_dataset, BIOSIGNAL_FEATURE_COLS, QUESTIONNAIRE_COLS
from src.preprocessing import drop_missing, impute_missing, normalise_features
from src.features import (
    compute_delta_features,
    select_features_variance,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = load_dataset('../data/raw')
df = drop_missing(df)
df = impute_missing(df)
print(df.shape)

In [ ]:
# Variance-based feature selection
high_var_cols = select_features_variance(df, BIOSIGNAL_FEATURE_COLS, threshold=0.01)
print(f'Features passing variance threshold: {len(high_var_cols)}')

In [ ]:
# Delta features: puzzle − pre
delta_df = compute_delta_features(df, high_var_cols)
delta_cols = [c for c in delta_df.columns if c.endswith('_delta')]
print(f'Delta columns: {len(delta_cols)}')
delta_df[delta_cols].describe()

In [ ]:
# PCA on normalised biosignal features
df_norm = normalise_features(df, high_var_cols)
X = df_norm[high_var_cols].dropna()

pca = PCA(n_components=10)
components = pca.fit_transform(X)

plt.figure(figsize=(8, 4))
plt.bar(range(1, 11), pca.explained_variance_ratio_ * 100)
plt.xlabel('Principal component')
plt.ylabel('Explained variance (%)')
plt.title('PCA – Explained variance per component')
plt.tight_layout()
plt.savefig('../results/figures/pca_variance.png', dpi=150)
plt.show()